# Apex Submission Notebook

**Stage:** Apex  
**Purpose:** safer, grounded lift beyond Velocity without jumping straight into broad Supernova research.

This notebook starts from the Velocity recipe and adds the Apex core: constrained reblending, shrunken residual meta harvesting, diagnostic-only segment blends, international-specific signal patches, paired bootstrap checks, and guarded final submission writing.

Protected public baseline: Velocity `0.41697`. Protected local selected OOF: `0.4165458568`.


In [ ]:
from pathlib import Path
import sys
import gc
import json
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import GroupKFold

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import (
    compute_ndcg_fast,
    load_test,
    load_train,
    make_submission,
    train_val_split,
)
from src.final_pipeline import (
    build_test_feature_frame,
    build_validation_feature_frames,
    get_feature_cols,
    make_relevance,
    minmax_normalize_per_search,
    optimize_blend_weights,
    optimize_signed_blend_weights,
    rrf_score,
    save_run_json,
    sort_for_group_model,
    z_normalize_per_search,
)

try:
    from catboost import CatBoostRanker, Pool
except Exception as exc:
    CatBoostRanker = None
    Pool = None
    print(f"CatBoost unavailable: {exc}")

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
pd.set_option("display.max_columns", 240)
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)


## Configuration

Run validation first. Keep `RUN_FINAL_SUBMISSION=False` until the Apex gate cell selects a candidate with enough evidence. Set it to `True` only when you deliberately want to retrain on full data and write the CSV.


In [ ]:
RANDOM_STATE = 42
USE_SAMPLE = False
SAMPLE_FRAC = 0.15
SVD_COMPONENTS = 20

VALIDATION_TAG = "apex"
VALIDATION_RUN_NAME = f"{VALIDATION_TAG}_validation_run.json"
VALIDATION_PREDICTIONS_NAME = f"{VALIDATION_TAG}_validation_predictions.parquet"
FINAL_SUBMISSION_NAME = "submission_apex_primary.csv"

VELOCITY_PROTECTED_PUBLIC = 0.41697
VELOCITY_PROTECTED_OOF = 0.41654585679643447
VELOCITY_SIGNED_Z_DIAGNOSTIC = 0.4169677160581034

TRY_LGB_GPU = True
LGB_AUTO_FALLBACK_TO_CPU = True
GAIN_POLICY = "linear"

# Apex is evidence-gated, not time-gated.
RUN_POINTWISE = True
RUN_CLICK = True
RUN_LAMBDARANK = True
RUN_LAMBDARANK_TRUNC5 = True
RUN_XENDCG_SEED_AVG = True
RUN_XENDCG_NEGATIVE_DROPOUT = True
RUN_CATBOOST = CatBoostRanker is not None
RUN_META_STACKER = True
RUN_SEGMENT_BLEND = True  # diagnostic-only unless rank-diversity gates pass
RUN_FINAL_SUBMISSION = False

XENDCG_SEEDS = [42, 2026, 777, 1337, 9001]
XENDCG_NEGATIVE_DROPOUT_FRAC = 0.35
SNAPSHOT_MULTIPLIERS = (0.92, 0.96, 1.00)
USE_SNAPSHOT_PREDICTIONS = True
TARGET_PRIOR_SPLITS = 5

META_N_SPLITS = 10
META_NUM_BOOST_ROUND = 3000
META_EARLY_STOPPING = 120
META_ROUTING_MODE = "trimmed"
SEGMENT_BLEND_COL = "is_domestic"
SEGMENT_SHRINKAGE = 0.25
RESIDUAL_META_ALPHAS = [0.03, 0.05, 0.08, 0.12, 0.16, 0.20, 0.25, 0.30]

# Apex gates are relative to Velocity, not Overdrive.
SUBMISSION_GATE = VELOCITY_PROTECTED_OOF + 0.00045
STRONG_SUBMISSION_GATE = VELOCITY_PROTECTED_OOF + 0.00075
RESIDUAL_META_MIN_GAIN = 0.00050
SEGMENT_BLEND_MIN_GAIN = 0.00025
SEGMENT_SPEARMAN_MAX = 0.995
SEGMENT_TOP5_OVERLAP_MAX = 0.990
BOOTSTRAP_MIN_P_POSITIVE = 0.90
LOMO_DROP_TOLERANCE = 0.00020
ALLOW_DIAGNOSTIC_SUBMISSION = False
OVERWRITE_EXISTING_SUBMISSION = False
USE_TRAIN_TEST_COUNTS = False

PW_NUM_BOOST_ROUND = 9000
PW_EARLY_STOPPING = 150
CLICK_NUM_BOOST_ROUND = 6000
CLICK_EARLY_STOPPING = 150
LR_NUM_BOOST_ROUND = 10000
LR_EARLY_STOPPING = 250
LR_TRUNC5_NUM_BOOST_ROUND = 10000
LR_TRUNC5_EARLY_STOPPING = 250
XENDCG_NUM_BOOST_ROUND = 8000
XENDCG_EARLY_STOPPING = 250
CATBOOST_TASK_TYPE = "GPU"
CATBOOST_ITERATIONS = 9000
CATBOOST_EARLY_STOPPING = 400
CATBOOST_KEEP_SVD_FACTORS = False
CATBOOST_USE_ID_CATEGORICALS = True


## Parameters And Shared Helpers


In [ ]:
def load_run_json(name):
    path = OUTPUT_DIR / name
    if not path.exists():
        return {}
    with path.open() as f:
        return json.load(f)


def sanitize_lgb_params(params, objective, metric=None):
    clean = dict(params)
    for key in [
        "best_iteration", "best_score", "val_ndcg", "val_logloss", "notes",
        "feature_importance", "best_params",
    ]:
        clean.pop(key, None)
    clean["objective"] = objective
    if metric is not None:
        clean["metric"] = metric
    clean.setdefault("feature_pre_filter", False)
    clean.setdefault("verbose", -1)
    clean.setdefault("seed", RANDOM_STATE)
    clean.setdefault("num_threads", -1)
    return clean


def apply_lgb_gain_policy(params):
    clean = dict(params)
    clean.pop("label_gain", None)
    if GAIN_POLICY == "linear":
        clean["label_gain"] = [0, 1, 2, 3, 4, 5]
    elif GAIN_POLICY not in {"default", "default_exponential"}:
        raise ValueError(f"Unknown GAIN_POLICY={GAIN_POLICY!r}")
    return clean


def with_lgb_seed(params, seed):
    seeded = dict(params)
    seeded.update({
        "seed": int(seed),
        "bagging_seed": int(seed) + 11,
        "feature_fraction_seed": int(seed) + 17,
        "data_random_seed": int(seed) + 23,
    })
    return seeded


def train_lgb(params, train_set, **kwargs):
    try:
        return lgb.train(params, train_set, **kwargs)
    except lgb.basic.LightGBMError as exc:
        if params.get("device") == "gpu" and LGB_AUTO_FALLBACK_TO_CPU:
            cpu_params = dict(params)
            cpu_params.pop("device", None)
            cpu_params.pop("gpu_use_dp", None)
            cpu_params["device_type"] = "cpu"
            print(f"LightGBM GPU failed; retrying on CPU: {exc}")
            return lgb.train(cpu_params, train_set, **kwargs)
        raise


def lgb_iteration_count(model, fallback):
    if model is None:
        return None
    best_iter = getattr(model, "best_iteration", None)
    if best_iter is not None and int(best_iter) > 0:
        return int(best_iter)
    current_iter = model.current_iteration() if hasattr(model, "current_iteration") else None
    if current_iter is not None and int(current_iter) > 0:
        return int(current_iter)
    return int(fallback)


def catboost_iteration_count(model, fallback):
    if model is None:
        return None
    best_iter = model.get_best_iteration()
    if best_iter is None or int(best_iter) < 0:
        return int(fallback)
    return int(best_iter) + 1


def iteration_or_fallback(value, fallback):
    if value is None:
        return int(fallback)
    value = int(value)
    return value if value > 0 else int(fallback)


def predict_lgb_snapshot(model, X, fallback_rounds, use_snapshots=USE_SNAPSHOT_PREDICTIONS):
    best_iter = lgb_iteration_count(model, fallback_rounds)
    if not use_snapshots:
        return model.predict(X, num_iteration=best_iter)
    checkpoints = sorted({max(1, int(round(best_iter * m))) for m in SNAPSHOT_MULTIPLIERS})
    preds = [model.predict(X, num_iteration=cp) for cp in checkpoints]
    return np.mean(preds, axis=0)


def add_score_normalizations(frame, score_cols):
    frame = frame.copy()
    for col in score_cols:
        frame[f"{col}_z"] = z_normalize_per_search(frame, col)
        frame[f"{col}_mm"] = minmax_normalize_per_search(frame, col)
        frame[f"{col}_rank"] = frame.groupby("srch_id")[col].rank(
            ascending=False, pct=True, method="first"
        ).astype("float32")
    return frame


gpu_params = {"device": "gpu", "gpu_use_dp": False} if TRY_LGB_GPU else {}
print("LightGBM device params:", gpu_params)
print("LightGBM rank gain policy:", GAIN_POLICY)

pw_run = load_run_json("pointwise_run2.json")
lr_run = load_run_json("lambdarank_run2.json")

PW_PARAMS = sanitize_lgb_params(
    pw_run.get("best_params", {
        "objective": "binary",
        "metric": "binary_logloss",
        "num_leaves": 511,
        "learning_rate": 0.01,
        "min_child_samples": 200,
        "lambda_l1": 0.1,
        "lambda_l2": 1.0,
        "min_gain_to_split": 0.01,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
    }),
    objective="binary",
    metric="binary_logloss",
)
PW_PARAMS.update(gpu_params)

CLICK_PARAMS = sanitize_lgb_params(
    {
        "objective": "binary",
        "metric": "binary_logloss",
        "num_leaves": 255,
        "learning_rate": 0.025,
        "min_child_samples": 150,
        "lambda_l1": 0.05,
        "lambda_l2": 1.0,
        "feature_fraction": 0.75,
        "bagging_fraction": 0.85,
        "bagging_freq": 5,
    },
    objective="binary",
    metric="binary_logloss",
)
CLICK_PARAMS.update(gpu_params)

LR_PARAMS = sanitize_lgb_params(
    lr_run.get("best_params", {
        "objective": "lambdarank",
        "metric": "ndcg",
        "ndcg_eval_at": [5],
        "num_leaves": 511,
        "learning_rate": 0.01,
        "min_child_samples": 200,
        "lambda_l1": 0.1,
        "lambda_l2": 1.0,
        "min_gain_to_split": 0.01,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
    }),
    objective="lambdarank",
    metric="ndcg",
)
LR_PARAMS.update(gpu_params)
LR_PARAMS.update({"ndcg_eval_at": [5]})
LR_PARAMS = apply_lgb_gain_policy(LR_PARAMS)

LR_TRUNC5_PARAMS = dict(LR_PARAMS)
LR_TRUNC5_PARAMS.update({
    "lambdarank_truncation_level": 5,
    "seed": RANDOM_STATE + 505,
})

XENDCG_PARAMS = dict(LR_PARAMS)
XENDCG_PARAMS.update({"objective": "rank_xendcg", "metric": "ndcg"})
XENDCG_PARAMS = apply_lgb_gain_policy(XENDCG_PARAMS)

META_PARAMS = sanitize_lgb_params(
    {
        "objective": "rank_xendcg",
        "metric": "ndcg",
        "ndcg_eval_at": [5],
        "num_leaves": 7,
        "learning_rate": 0.01,
        "min_child_samples": 500,
        "lambda_l1": 0.0,
        "lambda_l2": 5.0,
        "feature_fraction": 0.70,
        "bagging_fraction": 0.85,
        "bagging_freq": 1,
    },
    objective="rank_xendcg",
    metric="ndcg",
)
META_PARAMS.update(gpu_params)
META_PARAMS.update({"ndcg_eval_at": [5]})
META_PARAMS = apply_lgb_gain_policy(META_PARAMS)

print("Pointwise params:", PW_PARAMS)
print("Click params:", CLICK_PARAMS)
print("LambdaRank params:", LR_PARAMS)
print("LambdaRank@5 params:", LR_TRUNC5_PARAMS)
print("RankXENDCG base params:", XENDCG_PARAMS)
print("Meta params:", META_PARAMS)


## Apex Feature Block

Apex keeps the proven Velocity feature block, then adds narrow international, distance-confidence, and conservative price/value residual features. Target-derived values are OOF on training rows and train-only mapped onto validation/test rows.


In [ ]:
TARGET_VELOCITY_COLS = [
    "click_to_book_rate_oof",
    "dest_window_book_prior",
    "dest_window_book_count",
    "country_pair_star_book_prior",
    "country_pair_star_ctr_prior",
    "country_pair_star_count",
]

META_ROUTING_CANDIDATES_TRIMMED = [
    "search_hotel_count",
    "search_price_cv",
    "price_usd_rank",
    "price_z_in_search",
    "prop_quality_score_rank",
    "quality_rank_minus_price_rank",
    "is_price_quality_pareto",
    "value_rank_product",
    "is_domestic",
    "is_family",
    "orig_distance_was_missing",
    "country_pair_star_count",
]

META_ROUTING_CANDIDATES_WIDE = META_ROUTING_CANDIDATES_TRIMMED + [
    "search_price_std",
    "search_brand_frac",
    "search_mean_star",
    "star_diff_from_search_mean",
    "search_promo_frac",
    "price_rank_per_star",
    "price_z_in_tier",
    "price_vs_tier_median",
    "is_last_minute",
    "is_advance_purchase",
    "is_leisure_search",
    "comp_advantage_rank",
    "hotel_ctr_vs_dest_avg",
    "hotel_book_vs_dest_avg",
    "svd_cosine_similarity",
]

META_ROUTING_CANDIDATES = (
    META_ROUTING_CANDIDATES_TRIMMED if META_ROUTING_MODE == "trimmed" else META_ROUTING_CANDIDATES_WIDE
)


def available(cols, frame):
    return [c for c in cols if c in frame.columns]


def _booking_window_bucket(window):
    return pd.cut(
        window,
        bins=[-1, 0, 1, 7, 14, 30, 90, 365, 9999],
        labels=[0, 1, 2, 3, 4, 5, 6, 7],
    ).astype("float32")


def _multiindex_map(frame, key_cols, mapping):
    keys = pd.MultiIndex.from_frame(frame[key_cols])
    return pd.Series(keys.map(mapping), index=frame.index)


def _ensure_velocity_keys(frame):
    frame = frame.copy()
    if "booking_window_bucket" not in frame.columns:
        frame["booking_window_bucket"] = _booking_window_bucket(frame["srch_booking_window"])
    return frame


def _apply_velocity_target_maps(fit_df, frame, m=50):
    fit_df = _ensure_velocity_keys(fit_df)
    frame = _ensure_velocity_keys(frame)
    out = frame.copy()

    click_sum_global = float(fit_df["click_bool"].sum())
    book_sum_global = float(fit_df["booking_bool"].sum())
    global_book = float(fit_df["booking_bool"].mean()) if len(fit_df) else 0.0
    global_click = float(fit_df["click_bool"].mean()) if len(fit_df) else 0.0
    global_ctb = book_sum_global / max(click_sum_global, 1.0)

    prop = fit_df.groupby("prop_id").agg(
        book_sum=("booking_bool", "sum"),
        click_sum=("click_bool", "sum"),
    )
    prop["click_to_book_rate_oof"] = (
        (prop["book_sum"] + m * global_ctb) / (prop["click_sum"] + m)
    ).astype("float32")
    out["click_to_book_rate_oof"] = (
        out["prop_id"].map(prop["click_to_book_rate_oof"]).fillna(global_ctb).astype("float32")
    )

    dest_key = ["srch_destination_id", "booking_window_bucket"]
    dest = fit_df.groupby(dest_key).agg(
        book_sum=("booking_bool", "sum"),
        count=("booking_bool", "size"),
    )
    dest["dest_window_book_prior"] = (
        (dest["book_sum"] + m * global_book) / (dest["count"] + m)
    ).astype("float32")
    out["dest_window_book_prior"] = _multiindex_map(
        out, dest_key, dest["dest_window_book_prior"]
    ).fillna(global_book).astype("float32")
    out["dest_window_book_count"] = _multiindex_map(
        out, dest_key, dest["count"]
    ).fillna(0).astype("int32")

    cps_key = ["visitor_location_country_id", "prop_country_id", "prop_starrating"]
    cps = fit_df.groupby(cps_key).agg(
        book_sum=("booking_bool", "sum"),
        click_sum=("click_bool", "sum"),
        count=("booking_bool", "size"),
    )
    cps["country_pair_star_book_prior"] = (
        (cps["book_sum"] + m * global_book) / (cps["count"] + m)
    ).astype("float32")
    cps["country_pair_star_ctr_prior"] = (
        (cps["click_sum"] + m * global_click) / (cps["count"] + m)
    ).astype("float32")
    out["country_pair_star_book_prior"] = _multiindex_map(
        out, cps_key, cps["country_pair_star_book_prior"]
    ).fillna(global_book).astype("float32")
    out["country_pair_star_ctr_prior"] = _multiindex_map(
        out, cps_key, cps["country_pair_star_ctr_prior"]
    ).fillna(global_click).astype("float32")
    out["country_pair_star_count"] = _multiindex_map(
        out, cps_key, cps["count"]
    ).fillna(0).astype("int32")
    return out


def add_velocity_target_features(train_df, holdout_df=None, n_splits=5):
    train_df = _ensure_velocity_keys(train_df)
    holdout_df = None if holdout_df is None else _ensure_velocity_keys(holdout_df)
    splits = min(n_splits, train_df["srch_id"].nunique())
    for col in TARGET_VELOCITY_COLS:
        train_df[col] = np.nan

    if splits < 2:
        train_df = _apply_velocity_target_maps(train_df, train_df)
    else:
        splitter = GroupKFold(n_splits=splits)
        for fit_idx, map_idx in splitter.split(train_df, groups=train_df["srch_id"]):
            fold_frame = _apply_velocity_target_maps(train_df.iloc[fit_idx], train_df.iloc[map_idx])
            train_df.loc[train_df.index[map_idx], TARGET_VELOCITY_COLS] = fold_frame[
                TARGET_VELOCITY_COLS
            ].to_numpy()
        for col in TARGET_VELOCITY_COLS:
            if train_df[col].isna().any():
                fallback = 0 if col.endswith("count") else train_df[col].mean()
                train_df[col] = train_df[col].fillna(fallback)

    if holdout_df is not None:
        holdout_df = _apply_velocity_target_maps(train_df, holdout_df)

    for target in [train_df, holdout_df]:
        if target is None:
            continue
        for col in TARGET_VELOCITY_COLS:
            if col.endswith("count"):
                target[col] = target[col].fillna(0).astype("int32")
            else:
                target[col] = target[col].astype("float32")
    return train_df, holdout_df


def add_velocity_reference_features(reference_df, frame):
    frame = frame.copy()
    ref = reference_df.copy()
    key = ["visitor_location_country_id", "prop_country_id"]
    global_dist = float(ref["orig_destination_distance"].median())
    dist_map = ref.groupby(key)["orig_destination_distance"].median()
    pair_median = _multiindex_map(frame, key, dist_map)
    frame["orig_distance_was_missing"] = frame["orig_destination_distance"].isna().astype("int8")
    frame["orig_distance_country_pair_median"] = pair_median.fillna(global_dist).astype("float32")
    frame["orig_distance_imputed"] = (
        frame["orig_destination_distance"].fillna(frame["orig_distance_country_pair_median"])
    ).fillna(global_dist).astype("float32")
    frame["log_distance_imputed"] = np.log1p(frame["orig_distance_imputed"]).astype("float32")

    vc_dest_key = ["visitor_location_country_id", "srch_destination_id"]
    vc_dest_count = ref.groupby(vc_dest_key).size().astype("int32")
    frame["visitor_dest_popularity"] = _multiindex_map(
        frame, vc_dest_key, vc_dest_count
    ).fillna(0).astype("int32")
    frame["log_visitor_dest_popularity"] = np.log1p(frame["visitor_dest_popularity"]).astype("float32")
    return frame


def _svd_factor_cols(frame, prefix):
    cols = [c for c in frame.columns if c.startswith(prefix)]
    return sorted(cols, key=lambda c: int(c.rsplit("f", 1)[-1]))


def _add_price_quality_pareto(frame):
    frame = frame.copy()
    if not {"price_per_night", "prop_quality_score"}.issubset(frame.columns):
        frame["is_price_quality_pareto"] = np.int8(0)
        return frame
    pareto = pd.Series(0, index=frame.index, dtype="int8")
    for _, group in frame.groupby("srch_id", sort=False):
        price = group["price_per_night"].fillna(np.inf).to_numpy()
        quality = group["prop_quality_score"].fillna(-np.inf).to_numpy()
        order = np.lexsort((-quality, price))
        best_quality = -np.inf
        group_index = group.index.to_numpy()
        for pos in order:
            if quality[pos] >= best_quality:
                pareto.loc[group_index[pos]] = 1
                best_quality = max(best_quality, quality[pos])
    frame["is_price_quality_pareto"] = pareto.astype("int8")
    return frame


def add_velocity_direct_features(frame):
    frame = frame.copy()
    dest_cols = _svd_factor_cols(frame, "svd_dest_f")
    hotel_cols = _svd_factor_cols(frame, "svd_hotel_f")
    if dest_cols and len(dest_cols) == len(hotel_cols):
        dest = frame[dest_cols].to_numpy(dtype="float32")
        hotel = frame[hotel_cols].to_numpy(dtype="float32")
        dot = (dest * hotel).sum(axis=1)
        denom = np.linalg.norm(dest, axis=1) * np.linalg.norm(hotel, axis=1)
        frame["svd_cosine_similarity"] = (dot / np.where(denom > 1e-8, denom, 1.0)).astype("float32")

    if {"hotel_ctr_unbiased", "dest_star_ctr_unbiased"}.issubset(frame.columns):
        frame["hotel_ctr_vs_dest_avg"] = (
            frame["hotel_ctr_unbiased"] - frame["dest_star_ctr_unbiased"]
        ).astype("float32")
    if {"hotel_book_rate_unbiased", "dest_star_book_rate_unbiased"}.issubset(frame.columns):
        frame["hotel_book_vs_dest_avg"] = (
            frame["hotel_book_rate_unbiased"] - frame["dest_star_book_rate_unbiased"]
        ).astype("float32")
    if {"prop_quality_score_rank", "price_usd_rank"}.issubset(frame.columns):
        frame["quality_rank_minus_price_rank"] = (
            frame["prop_quality_score_rank"] - frame["price_usd_rank"]
        ).astype("float32")
        frame["value_rank_product"] = (
            frame["prop_quality_score_rank"] * (1.0 - frame["price_usd_rank"])
        ).astype("float32")
    frame = _add_price_quality_pareto(frame)
    return frame


def add_velocity_features(train_df, holdout_df=None, n_splits=TARGET_PRIOR_SPLITS):
    train_df, holdout_df = add_velocity_target_features(train_df, holdout_df, n_splits=n_splits)
    train_df = add_velocity_reference_features(train_df, train_df)
    train_df = add_velocity_direct_features(train_df)
    if holdout_df is not None:
        holdout_df = add_velocity_reference_features(train_df, holdout_df)
        holdout_df = add_velocity_direct_features(holdout_df)
    return train_df, holdout_df


# ---------------------------------------------------------------------------
# Apex additions: narrow, validation-friendly patches on top of Velocity.
# ---------------------------------------------------------------------------
TARGET_APEX_COLS = TARGET_VELOCITY_COLS + [
    "country_pair_star_book_lift_over_prop_country_star",
    "country_pair_star_book_lift_over_visitor_country",
    "country_pair_support_log",
    "country_pair_is_sparse",
    "intl_prior_confidence",
]

APEX_ROUTING_ADDITIONS = [
    "country_pair_star_book_lift_over_prop_country_star",
    "country_pair_star_book_lift_over_visitor_country",
    "country_pair_support_log",
    "country_pair_is_sparse",
    "intl_prior_confidence",
    "visitor_dest_pop_lift_over_global_dest",
    "visitor_dest_support_log",
    "orig_dist_pair_median_support",
    "orig_dist_imputed_confidence",
    "orig_dist_rank_in_search",
    "orig_dist_z_in_search",
    "orig_dist_vs_country_pair_median",
    "orig_dist_missing_x_international",
    "price_resid_site_country",
    "price_resid_site_country_star",
    "price_resid_country_los",
    "price_residual_rank_in_search",
    "value_residual",
    "intl_price_residual",
]

META_ROUTING_CANDIDATES_TRIMMED = list(dict.fromkeys(
    META_ROUTING_CANDIDATES_TRIMMED + APEX_ROUTING_ADDITIONS[:12]
))
META_ROUTING_CANDIDATES_WIDE = list(dict.fromkeys(
    META_ROUTING_CANDIDATES_WIDE + APEX_ROUTING_ADDITIONS
))
META_ROUTING_CANDIDATES = (
    META_ROUTING_CANDIDATES_TRIMMED if META_ROUTING_MODE == "trimmed" else META_ROUTING_CANDIDATES_WIDE
)


def _book_prior_table(fit_df, key_cols, m=75):
    global_book = float(fit_df["booking_bool"].mean()) if len(fit_df) else 0.0
    agg = fit_df.groupby(key_cols).agg(
        book_sum=("booking_bool", "sum"),
        count=("booking_bool", "size"),
    )
    agg["_book_prior"] = ((agg["book_sum"] + m * global_book) / (agg["count"] + m)).astype("float32")
    return agg, global_book


def _apply_apex_target_maps(fit_df, frame, m=75):
    fit_df = _ensure_velocity_keys(fit_df)
    out = _apply_velocity_target_maps(fit_df, frame, m=m)
    out = _ensure_velocity_keys(out)

    cps_support = out["country_pair_star_count"].fillna(0).astype("float32")
    out["country_pair_support_log"] = np.log1p(cps_support).astype("float32")
    out["country_pair_is_sparse"] = (cps_support < 20).astype("int8")

    pc_key = ["prop_country_id", "prop_starrating"]
    pc_table, global_book = _book_prior_table(fit_df, pc_key, m=m)
    pc_prior = _multiindex_map(out, pc_key, pc_table["_book_prior"]).fillna(global_book).astype("float32")

    vc_key = ["visitor_location_country_id"]
    vc_table, _ = _book_prior_table(fit_df, vc_key, m=m)
    vc_prior = out["visitor_location_country_id"].map(vc_table["_book_prior"]).fillna(global_book).astype("float32")

    out["country_pair_star_book_lift_over_prop_country_star"] = (
        out["country_pair_star_book_prior"] - pc_prior
    ).astype("float32")
    out["country_pair_star_book_lift_over_visitor_country"] = (
        out["country_pair_star_book_prior"] - vc_prior
    ).astype("float32")
    is_intl = (out["visitor_location_country_id"] != out["prop_country_id"]).astype("float32")
    out["intl_prior_confidence"] = (
        is_intl * (1.0 - np.exp(-cps_support / 50.0))
    ).astype("float32")
    return out


def add_apex_target_features(train_df, holdout_df=None, n_splits=TARGET_PRIOR_SPLITS):
    train_df = _ensure_velocity_keys(train_df)
    holdout_df = None if holdout_df is None else _ensure_velocity_keys(holdout_df)
    splits = min(n_splits, train_df["srch_id"].nunique())
    for col in TARGET_APEX_COLS:
        train_df[col] = np.nan

    if splits < 2:
        train_df = _apply_apex_target_maps(train_df, train_df)
    else:
        splitter = GroupKFold(n_splits=splits)
        for fit_idx, map_idx in splitter.split(train_df, groups=train_df["srch_id"]):
            fold_frame = _apply_apex_target_maps(train_df.iloc[fit_idx], train_df.iloc[map_idx])
            train_df.loc[train_df.index[map_idx], TARGET_APEX_COLS] = fold_frame[TARGET_APEX_COLS].to_numpy()
        for col in TARGET_APEX_COLS:
            if train_df[col].isna().any():
                fallback = 0 if col.endswith(("count", "sparse")) else train_df[col].mean()
                train_df[col] = train_df[col].fillna(fallback)

    if holdout_df is not None:
        holdout_df = _apply_apex_target_maps(train_df, holdout_df)

    for target in [train_df, holdout_df]:
        if target is None:
            continue
        for col in TARGET_APEX_COLS:
            if col.endswith(("count", "sparse")):
                target[col] = target[col].fillna(0).astype("int32")
            else:
                target[col] = target[col].replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
    return train_df, holdout_df


def _group_median_map(reference_df, key_cols, value_col):
    values = reference_df[value_col].replace([np.inf, -np.inf], np.nan)
    return reference_df.assign(_value=values).groupby(key_cols)["_value"].median()


def add_apex_reference_features(reference_df, frame):
    frame = frame.copy()
    ref = reference_df.copy()
    is_intl = (frame["visitor_location_country_id"] != frame["prop_country_id"]).astype("float32")

    # International destination demand and support, target-free.
    vc_dest_key = ["visitor_location_country_id", "srch_destination_id"]
    vc_dest_count = ref.groupby(vc_dest_key).size().astype("float32")
    dest_count = ref.groupby("srch_destination_id").size().astype("float32")
    vc_dest = _multiindex_map(frame, vc_dest_key, vc_dest_count).fillna(0).astype("float32")
    global_dest = frame["srch_destination_id"].map(dest_count).fillna(0).astype("float32")
    frame["visitor_dest_support_log"] = np.log1p(vc_dest).astype("float32")
    frame["visitor_dest_pop_lift_over_global_dest"] = (
        (vc_dest + 1.0) / (global_dest + 1.0)
    ).astype("float32")

    # Distance imputation confidence and relative context.
    dist_key = ["visitor_location_country_id", "prop_country_id"]
    dist_agg = ref.groupby(dist_key)["orig_destination_distance"].agg(["median", "count"])
    pair_median = _multiindex_map(frame, dist_key, dist_agg["median"]).fillna(frame.get("orig_distance_country_pair_median", np.nan))
    pair_support = _multiindex_map(frame, dist_key, dist_agg["count"]).fillna(0).astype("float32")
    frame["orig_dist_pair_median_support"] = pair_support.astype("float32")
    frame["orig_dist_imputed_confidence"] = (1.0 - np.exp(-pair_support / 20.0)).astype("float32")
    frame["orig_dist_vs_country_pair_median"] = (
        np.log1p(frame["orig_distance_imputed"].clip(lower=0)) - np.log1p(pair_median.clip(lower=0))
    ).replace([np.inf, -np.inf], 0).fillna(0).astype("float32")
    g_dist = frame.groupby("srch_id")["orig_distance_imputed"]
    dist_std = g_dist.transform("std").replace(0, 1).fillna(1)
    frame["orig_dist_rank_in_search"] = g_dist.rank(pct=True).astype("float32")
    frame["orig_dist_z_in_search"] = (
        (frame["orig_distance_imputed"] - g_dist.transform("mean")) / dist_std
    ).replace([np.inf, -np.inf], 0).fillna(0).astype("float32")
    frame["orig_dist_missing_x_international"] = (
        frame["orig_distance_was_missing"].astype("float32") * is_intl
    ).astype("float32")

    # Conservative price/value residuals. These use only observable price/search context.
    if "price_per_night" not in ref.columns:
        ref["price_per_night"] = ref["price_usd"] / ref["srch_length_of_stay"].clip(lower=1)
    if "price_per_night" not in frame.columns:
        frame["price_per_night"] = frame["price_usd"] / frame["srch_length_of_stay"].clip(lower=1)
    ref["log_price_per_night"] = np.log1p(ref["price_per_night"].clip(lower=0))
    frame["log_price_per_night"] = np.log1p(frame["price_per_night"].clip(lower=0)).astype("float32")

    price_specs = {
        "site_country": ["site_id", "prop_country_id"],
        "site_country_star": ["site_id", "prop_country_id", "prop_starrating"],
        "country_los": ["prop_country_id", "srch_length_of_stay"],
    }
    global_log_price = float(ref["log_price_per_night"].median())
    for suffix, keys in price_specs.items():
        med = _group_median_map(ref, keys, "log_price_per_night")
        baseline = _multiindex_map(frame, keys, med).fillna(global_log_price)
        frame[f"price_resid_{suffix}"] = (
            frame["log_price_per_night"] - baseline
        ).replace([np.inf, -np.inf], 0).fillna(0).astype("float32")

    resid_col = "price_resid_site_country"
    frame["price_residual_rank_in_search"] = (
        frame.groupby("srch_id")[resid_col].rank(pct=True)
    ).astype("float32")
    frame["price_residual_x_starrating"] = (
        frame[resid_col] * frame["prop_starrating"]
    ).astype("float32")
    if {"prop_quality_score_rank", "price_residual_rank_in_search"}.issubset(frame.columns):
        frame["value_residual"] = (
            frame["prop_quality_score_rank"] - frame["price_residual_rank_in_search"]
        ).astype("float32")
    else:
        frame["value_residual"] = np.float32(0.0)
    frame["intl_price_residual"] = (frame[resid_col] * is_intl).astype("float32")
    return frame


def add_apex_features(train_df, holdout_df=None, n_splits=TARGET_PRIOR_SPLITS):
    train_df, holdout_df = add_apex_target_features(train_df, holdout_df, n_splits=n_splits)

    train_df = add_velocity_reference_features(train_df, train_df)
    if holdout_df is not None:
        holdout_df = add_velocity_reference_features(train_df, holdout_df)

    train_df = add_velocity_direct_features(train_df)
    if holdout_df is not None:
        holdout_df = add_velocity_direct_features(holdout_df)

    train_df = add_apex_reference_features(train_df, train_df)
    if holdout_df is not None:
        holdout_df = add_apex_reference_features(train_df, holdout_df)
    return train_df, holdout_df


## Training And Ensemble Helpers


In [ ]:
def train_binary_model(name, params, train_label, val_label, num_boost_round, early_stopping):
    train_set = lgb.Dataset(train_df[feature_cols], label=train_label)
    val_set = lgb.Dataset(val_df[feature_cols], label=val_label, reference=train_set)
    model = train_lgb(
        params,
        train_set,
        num_boost_round=num_boost_round,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(early_stopping), lgb.log_evaluation(200)],
    )
    pred_df[name] = predict_lgb_snapshot(model, val_df[feature_cols], num_boost_round)
    ndcg = compute_ndcg_fast(pred_df, name)
    print(f"{name} NDCG@5: {ndcg:.6f}  trees={lgb_iteration_count(model, num_boost_round)}")
    return model, ndcg


def predict_sorted_model(model, sorted_frame, X, score_col, fallback_rounds):
    pred = sorted_frame[["srch_id", "prop_id"]].copy()
    pred[score_col] = predict_lgb_snapshot(model, X, fallback_rounds)
    return pred


def query_local_negative_dropout(frame, drop_frac, seed):
    if drop_frac <= 0:
        return frame
    rng = np.random.default_rng(seed)
    rel = make_relevance(frame)
    keep = rel > 0
    zero_idx = np.where(~keep)[0]
    keep[zero_idx] = rng.random(len(zero_idx)) >= drop_frac
    keep_series = pd.Series(keep, index=frame.index)
    per_query_kept = keep_series.groupby(frame["srch_id"]).sum()
    empty_queries = set(per_query_kept.index[per_query_kept == 0])
    if empty_queries:
        first_rows = frame[frame["srch_id"].isin(empty_queries)].groupby("srch_id", sort=False).head(1).index
        keep_series.loc[first_rows] = True
    return frame.loc[keep_series.to_numpy()].copy()


def train_rank_variant(score_col, params, num_boost_round, early_stopping, seed=None, use_dropout=False):
    fit_df = train_df
    if use_dropout:
        fit_df = query_local_negative_dropout(train_df, XENDCG_NEGATIVE_DROPOUT_FRAC, seed or RANDOM_STATE)
        print(f"{score_col}: query-local dropout kept {len(fit_df):,}/{len(train_df):,} rows")
    tr_sorted, X_tr, y_tr, group_tr = sort_for_group_model(fit_df, feature_cols, make_relevance(fit_df))
    va_sorted, X_va, y_va, group_va = sort_for_group_model(val_df, feature_cols, make_relevance(val_df))
    train_set = lgb.Dataset(X_tr, label=y_tr, group=group_tr)
    val_set = lgb.Dataset(X_va, label=y_va, group=group_va, reference=train_set)
    model = train_lgb(
        params,
        train_set,
        num_boost_round=num_boost_round,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(early_stopping), lgb.log_evaluation(200)],
    )
    pred = predict_sorted_model(model, va_sorted, X_va, score_col, num_boost_round)
    ndcg_frame = pred_df[["srch_id", "prop_id", "booking_bool", "click_bool"]].merge(
        pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one"
    )
    ndcg = compute_ndcg_fast(ndcg_frame, score_col)
    print(f"{score_col} NDCG@5: {ndcg:.6f}  trees={lgb_iteration_count(model, num_boost_round)}")
    return model, pred, ndcg


def evaluate_oof_blend(frame, normalized_score_cols, signed=False, n_splits=5):
    groups = frame["srch_id"].to_numpy()
    splits = min(n_splits, frame["srch_id"].nunique())
    if splits < 2:
        return np.nan, None
    oof = np.zeros(len(frame), dtype="float32")
    fold_weights = []
    splitter = GroupKFold(n_splits=splits)
    for fit_idx, hold_idx in splitter.split(frame, groups=groups):
        fit_frame = frame.iloc[fit_idx]
        hold_frame = frame.iloc[hold_idx]
        if signed:
            fold_w, _ = optimize_signed_blend_weights(fit_frame, normalized_score_cols, maxiter=250)
        else:
            fold_w, _ = optimize_blend_weights(fit_frame, normalized_score_cols, maxiter=250)
        fold_weights.append(fold_w)
        blend = np.zeros(len(hold_frame), dtype="float64")
        for weight, col in zip(fold_w, normalized_score_cols):
            blend += weight * hold_frame[col].to_numpy(dtype="float64")
        oof[hold_idx] = blend.astype("float32")
    tmp = frame[["srch_id", "booking_bool", "click_bool"]].copy()
    tmp["_oof_blend"] = oof
    return compute_ndcg_fast(tmp, "_oof_blend"), fold_weights


def prepare_meta_frame(frame, score_cols, routing_cols):
    meta = add_score_normalizations(frame.reset_index(drop=True), score_cols)
    norm_cols = []
    for col in score_cols:
        norm_cols.extend([f"{col}_z", f"{col}_mm", f"{col}_rank"])
    meta_feature_cols = norm_cols + routing_cols
    meta[meta_feature_cols] = meta[meta_feature_cols].replace([np.inf, -np.inf], np.nan)
    return meta, meta_feature_cols


def train_meta_oof(meta_df, meta_feature_cols):
    splits = min(META_N_SPLITS, meta_df["srch_id"].nunique())
    if splits < 2:
        raise ValueError("Need at least two search groups for meta OOF stacking")
    oof = np.zeros(len(meta_df), dtype="float32")
    best_iters = []
    fold_scores = []
    splitter = GroupKFold(n_splits=splits)
    for fold, (tr_idx, va_idx) in enumerate(splitter.split(meta_df, groups=meta_df["srch_id"]), start=1):
        fold_train = meta_df.iloc[tr_idx].copy()
        fold_val = meta_df.iloc[va_idx].copy()
        fold_val["_row_id"] = fold_val.index
        tr_sorted, X_tr, y_tr, group_tr = sort_for_group_model(fold_train, meta_feature_cols, make_relevance(fold_train))
        va_sorted, X_va, y_va, group_va = sort_for_group_model(fold_val, meta_feature_cols, make_relevance(fold_val))
        tr_set = lgb.Dataset(X_tr, label=y_tr, group=group_tr)
        va_set = lgb.Dataset(X_va, label=y_va, group=group_va, reference=tr_set)
        model = train_lgb(
            with_lgb_seed(META_PARAMS, RANDOM_STATE + fold),
            tr_set,
            num_boost_round=META_NUM_BOOST_ROUND,
            valid_sets=[va_set],
            callbacks=[lgb.early_stopping(META_EARLY_STOPPING), lgb.log_evaluation(0)],
        )
        preds = model.predict(X_va, num_iteration=lgb_iteration_count(model, META_NUM_BOOST_ROUND))
        oof[va_sorted["_row_id"].to_numpy()] = preds.astype("float32")
        best_iters.append(lgb_iteration_count(model, META_NUM_BOOST_ROUND))
        fold_eval = fold_val[["srch_id", "booking_bool", "click_bool"]].copy()
        fold_eval["_meta"] = oof[fold_val.index]
        fold_score = compute_ndcg_fast(fold_eval, "_meta")
        fold_scores.append(fold_score)
        print(f"Meta fold {fold}: NDCG@5={fold_score:.6f}, trees={best_iters[-1]}")
    meta_eval = meta_df[["srch_id", "booking_bool", "click_bool"]].copy()
    meta_eval["meta_oof_score"] = oof
    oof_score = compute_ndcg_fast(meta_eval, "meta_oof_score")
    full_rounds = int(np.median(best_iters)) if best_iters else 300
    all_sorted, X_all, y_all, group_all = sort_for_group_model(meta_df, meta_feature_cols, make_relevance(meta_df))
    full_model = train_lgb(
        with_lgb_seed(META_PARAMS, RANDOM_STATE + 999),
        lgb.Dataset(X_all, label=y_all, group=group_all),
        num_boost_round=full_rounds,
    )
    return oof, oof_score, full_model, full_rounds, fold_scores


def segment_blend_oof(frame, z_cols, segment_col, shrinkage=0.30, n_splits=5):
    if segment_col not in frame.columns:
        return np.nan, None, None
    splits = min(n_splits, frame["srch_id"].nunique())
    if splits < 2:
        return np.nan, None, None
    oof = np.zeros(len(frame), dtype="float32")
    configs = []
    splitter = GroupKFold(n_splits=splits)
    for fit_idx, hold_idx in splitter.split(frame, groups=frame["srch_id"]):
        fit_frame = frame.iloc[fit_idx]
        hold_frame = frame.iloc[hold_idx]
        global_w, _ = optimize_signed_blend_weights(fit_frame, z_cols, maxiter=250)
        segment_weights = {}
        for value in sorted(fit_frame[segment_col].dropna().unique()):
            fit_seg = fit_frame[fit_frame[segment_col] == value]
            if fit_seg["srch_id"].nunique() < 10:
                seg_w = global_w
            else:
                raw_w, _ = optimize_signed_blend_weights(fit_seg, z_cols, maxiter=250)
                seg_w = (1.0 - shrinkage) * global_w + shrinkage * raw_w
                denom = np.abs(seg_w).sum()
                seg_w = global_w if denom < 1e-12 else seg_w / denom
            segment_weights[str(value)] = seg_w
        configs.append({
            "global": global_w,
            "segments": segment_weights,
            "segment_col": segment_col,
            "shrinkage": shrinkage,
        })
        for value, sub_idx in hold_frame.groupby(segment_col).groups.items():
            idx = np.asarray(list(sub_idx))
            weights = segment_weights.get(str(value), global_w)
            sub = frame.loc[idx]
            blend = np.zeros(len(sub), dtype="float64")
            for weight, col in zip(weights, z_cols):
                blend += weight * sub[col].to_numpy(dtype="float64")
            oof[idx] = blend.astype("float32")
    tmp = frame[["srch_id", "booking_bool", "click_bool"]].copy()
    tmp["segment_signed_z_blend"] = oof
    score = compute_ndcg_fast(tmp, "segment_signed_z_blend")
    full_config = fit_segment_blend(frame, z_cols, segment_col, shrinkage)
    return score, full_config, configs


def fit_segment_blend(frame, z_cols, segment_col, shrinkage=0.30):
    global_w, _ = optimize_signed_blend_weights(frame, z_cols, maxiter=500)
    segment_weights = {}
    for value in sorted(frame[segment_col].dropna().unique()):
        seg = frame[frame[segment_col] == value]
        if seg["srch_id"].nunique() < 10:
            weights = global_w
        else:
            raw_w, _ = optimize_signed_blend_weights(seg, z_cols, maxiter=350)
            weights = (1.0 - shrinkage) * global_w + shrinkage * raw_w
            denom = np.abs(weights).sum()
            weights = global_w if denom < 1e-12 else weights / denom
        segment_weights[str(value)] = [float(x) for x in weights]
    return {
        "global": [float(x) for x in global_w],
        "segments": segment_weights,
        "segment_col": segment_col,
        "shrinkage": float(shrinkage),
        "z_cols": list(z_cols),
    }


def apply_segment_blend(frame, config):
    z_cols = list(config["z_cols"])
    segment_col = config["segment_col"]
    global_w = np.asarray(config["global"], dtype="float64")
    out = pd.Series(0.0, index=frame.index, dtype="float64")
    if segment_col not in frame.columns:
        for weight, col in zip(global_w, z_cols):
            out += weight * frame[col].to_numpy(dtype="float64")
        return out.astype("float32")
    for value, idx in frame.groupby(segment_col).groups.items():
        weights = np.asarray(config["segments"].get(str(value), global_w), dtype="float64")
        sub = frame.loc[idx]
        blend = np.zeros(len(sub), dtype="float64")
        for weight, col in zip(weights, z_cols):
            blend += weight * sub[col].to_numpy(dtype="float64")
        out.loc[idx] = blend
    return out.astype("float32")


def residual_meta_score(frame, base_col, meta_col, alpha):
    tmp = frame[["srch_id", base_col, meta_col]].copy()
    tmp["_base_z"] = z_normalize_per_search(tmp, base_col)
    tmp["_meta_z"] = z_normalize_per_search(tmp, meta_col)
    residual = (tmp["_meta_z"] - tmp["_base_z"]).clip(-3.0, 3.0)
    return (tmp["_base_z"] + alpha * residual).astype("float32")


def residual_meta_oof(frame, base_col, meta_col, alphas, n_splits=5):
    splits = min(n_splits, frame["srch_id"].nunique())
    if splits < 2:
        return np.nan, None, []
    oof = np.zeros(len(frame), dtype="float32")
    fold_alphas = []
    splitter = GroupKFold(n_splits=splits)
    for fit_idx, hold_idx in splitter.split(frame, groups=frame["srch_id"]):
        fit = frame.iloc[fit_idx].copy()
        hold = frame.iloc[hold_idx].copy()
        best_alpha = None
        best_score = -np.inf
        for alpha in alphas:
            fit["_residual"] = residual_meta_score(fit, base_col, meta_col, alpha)
            score = compute_ndcg_fast(fit, "_residual")
            if score > best_score:
                best_score = score
                best_alpha = alpha
        hold["_residual"] = residual_meta_score(hold, base_col, meta_col, best_alpha)
        oof[hold_idx] = hold["_residual"].to_numpy(dtype="float32")
        fold_alphas.append(float(best_alpha))
    tmp = frame[["srch_id", "booking_bool", "click_bool"]].copy()
    tmp["residual_meta_score"] = oof
    oof_score = compute_ndcg_fast(tmp, "residual_meta_score")
    full_scores = []
    for alpha in alphas:
        frame["_residual"] = residual_meta_score(frame, base_col, meta_col, alpha)
        full_scores.append((compute_ndcg_fast(frame, "_residual"), alpha))
    _, full_alpha = max(full_scores, key=lambda x: x[0])
    frame.drop(columns=["_residual"], inplace=True, errors="ignore")
    return oof_score, float(full_alpha), fold_alphas


def top5_overlap(frame, score_a, score_b):
    rank_a = frame.groupby("srch_id")[score_a].rank(ascending=False, method="first")
    rank_b = frame.groupby("srch_id")[score_b].rank(ascending=False, method="first")
    denom = frame.groupby("srch_id").size().clip(upper=5)
    overlap = frame.loc[(rank_a <= 5) & (rank_b <= 5)].groupby("srch_id").size()
    overlap = overlap.reindex(denom.index, fill_value=0)
    return float((overlap / denom).mean())


def segment_ndcg(frame, score_col, segment_cols):
    out = {}
    for col in segment_cols:
        if col not in frame.columns:
            continue
        search_values = frame.groupby("srch_id")[col].first()
        for value in [0, 1]:
            search_ids = search_values.index[search_values == value]
            if len(search_ids) == 0:
                continue
            seg = frame[frame["srch_id"].isin(search_ids)]
            out[f"{col}={value}"] = float(compute_ndcg_fast(seg, score_col))
    return out


## Build Validation Features


In [ ]:
raw_df = load_train(sample_frac=SAMPLE_FRAC if USE_SAMPLE else None, random_state=RANDOM_STATE)
print(f"Raw rows: {len(raw_df):,}  searches: {raw_df['srch_id'].nunique():,}")

train_raw, val_raw = train_val_split(raw_df, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train rows: {len(train_raw):,}  Val rows: {len(val_raw):,}")
del raw_df
gc.collect()

count_reference_frame = train_raw
train_df, val_df = build_validation_feature_frames(
    train_raw,
    val_raw,
    count_reference_frame=count_reference_frame,
    svd_components=SVD_COMPONENTS,
)
train_df, val_df = add_apex_features(train_df, val_df, n_splits=TARGET_PRIOR_SPLITS)
del train_raw, val_raw, count_reference_frame
gc.collect()

assert train_df.duplicated(["srch_id", "prop_id"]).sum() == 0
assert val_df.duplicated(["srch_id", "prop_id"]).sum() == 0
feature_cols = get_feature_cols(train_df)
routing_cols = available(META_ROUTING_CANDIDATES, val_df)
print(f"Features: {len(feature_cols)}")
print(f"Meta routing features: {len(routing_cols)} -> {routing_cols}")
print("Top NaN feature counts:")
print(train_df[feature_cols].isna().sum().sort_values(ascending=False).head(25))


## Train Base Models


In [ ]:
pred_base_cols = ["srch_id", "prop_id", "booking_bool", "click_bool"] + routing_cols
pred_df = val_df[pred_base_cols].copy()

pw_model = click_model = lr_model = lr_trunc5_model = cat_model = None
pw_ndcg = click_ndcg = lr_ndcg = lr_trunc5_ndcg = cat_ndcg = None
xendcg_models = {}
xendcg_seed_scores = {}

if RUN_POINTWISE:
    pw_model, pw_ndcg = train_binary_model(
        "pw_score",
        PW_PARAMS,
        train_df["booking_bool"].to_numpy(),
        val_df["booking_bool"].to_numpy(),
        PW_NUM_BOOST_ROUND,
        PW_EARLY_STOPPING,
    )

if RUN_CLICK:
    click_model, click_ndcg = train_binary_model(
        "click_score",
        CLICK_PARAMS,
        train_df["click_bool"].to_numpy(),
        val_df["click_bool"].to_numpy(),
        CLICK_NUM_BOOST_ROUND,
        CLICK_EARLY_STOPPING,
    )

if RUN_LAMBDARANK:
    lr_model, lr_pred, lr_ndcg = train_rank_variant(
        "lr_score",
        LR_PARAMS,
        LR_NUM_BOOST_ROUND,
        LR_EARLY_STOPPING,
    )
    pred_df = pred_df.merge(lr_pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")

if RUN_LAMBDARANK_TRUNC5:
    lr_trunc5_model, lr_trunc5_pred, lr_trunc5_ndcg = train_rank_variant(
        "lr_trunc5_score",
        LR_TRUNC5_PARAMS,
        LR_TRUNC5_NUM_BOOST_ROUND,
        LR_TRUNC5_EARLY_STOPPING,
    )
    pred_df = pred_df.merge(lr_trunc5_pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")

if RUN_XENDCG_SEED_AVG:
    seed_cols = []
    for seed in XENDCG_SEEDS:
        col = f"xendcg_s{seed}_score"
        params = with_lgb_seed(XENDCG_PARAMS, seed)
        model, seed_pred, seed_ndcg = train_rank_variant(
            col,
            params,
            XENDCG_NUM_BOOST_ROUND,
            XENDCG_EARLY_STOPPING,
            seed=seed,
            use_dropout=RUN_XENDCG_NEGATIVE_DROPOUT,
        )
        xendcg_models[seed] = model
        xendcg_seed_scores[col] = float(seed_ndcg)
        pred_df = pred_df.merge(seed_pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")
        seed_cols.append(col)
    z_seed_cols = []
    for col in seed_cols:
        z_col = f"{col}_seed_z"
        pred_df[z_col] = z_normalize_per_search(pred_df, col)
        z_seed_cols.append(z_col)
    pred_df["xendcg_avg_score"] = pred_df[z_seed_cols].mean(axis=1).astype("float32")
    xendcg_avg_ndcg = compute_ndcg_fast(pred_df, "xendcg_avg_score")
    print(f"RankXENDCG seed average NDCG@5: {xendcg_avg_ndcg:.6f}")
else:
    xendcg_avg_ndcg = None

if RUN_CATBOOST and CatBoostRanker is not None:
    cb_feature_cols = get_feature_cols(train_df, keep_svd_factors=CATBOOST_KEEP_SVD_FACTORS)
    cb_train, X_cb_train, y_cb_train, _ = sort_for_group_model(train_df, cb_feature_cols, make_relevance(train_df))
    cb_val, X_cb_val, y_cb_val, _ = sort_for_group_model(val_df, cb_feature_cols, make_relevance(val_df))
    cat_features_names = []
    if CATBOOST_USE_ID_CATEGORICALS:
        cat_features_names.extend(["prop_id", "srch_destination_id"])
    cat_features_names.extend(["site_id", "visitor_location_country_id", "prop_country_id"])
    cat_features_names = list(dict.fromkeys(cat_features_names))
    missing_cat_features = set(cat_features_names) - set(cb_feature_cols)
    assert not missing_cat_features, f"Missing CatBoost categoricals: {missing_cat_features}"
    cat_features_idx = [cb_feature_cols.index(c) for c in cat_features_names]
    cb_train_pool = Pool(
        X_cb_train,
        label=y_cb_train,
        group_id=cb_train["srch_id"].to_numpy(),
        cat_features=cat_features_idx,
        feature_names=cb_feature_cols,
    )
    cb_val_pool = Pool(
        X_cb_val,
        label=y_cb_val,
        group_id=cb_val["srch_id"].to_numpy(),
        cat_features=cat_features_idx,
        feature_names=cb_feature_cols,
    )
    cat_model = CatBoostRanker(
        loss_function="YetiRank",
        eval_metric="NDCG:top=5;type=Base",
        iterations=CATBOOST_ITERATIONS,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=RANDOM_STATE,
        task_type=CATBOOST_TASK_TYPE,
        verbose=200,
        allow_writing_files=False,
    )
    cat_model.fit(
        cb_train_pool,
        eval_set=cb_val_pool,
        early_stopping_rounds=CATBOOST_EARLY_STOPPING,
        use_best_model=True,
    )
    cb_pred = cb_val[["srch_id", "prop_id"]].copy()
    cb_pred["cb_score"] = cat_model.predict(cb_val_pool)
    pred_df = pred_df.merge(cb_pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")
    cat_ndcg = compute_ndcg_fast(pred_df, "cb_score")
    print(f"CatBoost NDCG@5: {cat_ndcg:.6f}  trees={catboost_iteration_count(cat_model, CATBOOST_ITERATIONS)}")
    del cb_train_pool, cb_val_pool, cb_train, cb_val, X_cb_train, X_cb_val, y_cb_train, y_cb_val
    gc.collect()
else:
    print("Skipping CatBoost")


## Blend, Segment Diagnostics, And Residual Meta

Apex keeps simple segment blends diagnostic-only unless they create actual rank movement. The stronger candidate is a conservative residual correction on top of a protected base blend.


In [ ]:
candidate_score_cols = [
    c for c in [
        "pw_score",
        "click_score",
        "lr_score",
        "lr_trunc5_score",
        "xendcg_avg_score",
        "cb_score",
    ]
    if c in pred_df.columns
]
candidate_base_scores = {col: compute_ndcg_fast(pred_df, col) for col in candidate_score_cols}
print("Base scores:")
for col, score in sorted(candidate_base_scores.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {col}: {score:.6f}")


def signed_oof_for_model_set(cols):
    tmp_cols = ["srch_id", "prop_id", "booking_bool", "click_bool"] + cols
    tmp = add_score_normalizations(pred_df[tmp_cols].copy(), cols)
    tmp_z_cols = [f"{col}_z" for col in cols]
    ndcg, _ = evaluate_oof_blend(tmp, tmp_z_cols, signed=True, n_splits=META_N_SPLITS)
    return float(ndcg)

score_cols = list(candidate_score_cols)
lomo_diagnostics = {}
if len(score_cols) >= 3:
    lomo_reference = signed_oof_for_model_set(score_cols)
    for candidate_col in ["click_score", "lr_score"]:
        if candidate_col not in score_cols or len(score_cols) <= 2:
            continue
        without_cols = [col for col in score_cols if col != candidate_col]
        without_score = signed_oof_for_model_set(without_cols)
        delta_removed = without_score - lomo_reference
        keep_model = delta_removed <= LOMO_DROP_TOLERANCE
        lomo_diagnostics[candidate_col] = {
            "signed_oof_with_model": float(lomo_reference),
            "signed_oof_without_model": float(without_score),
            "delta_when_removed": float(delta_removed),
            "drop_tolerance": float(LOMO_DROP_TOLERANCE),
            "kept": bool(keep_model),
        }
        action = "keep" if keep_model else "drop"
        print(
            f"LOMO {candidate_col}: without={without_score:.6f}, with={lomo_reference:.6f}, "
            f"delta_removed={delta_removed:+.6f}, tol={LOMO_DROP_TOLERANCE:.6f} -> {action}"
        )
        if not keep_model:
            score_cols = without_cols
            lomo_reference = without_score
else:
    lomo_reference = np.nan

base_scores = {col: candidate_base_scores[col] for col in score_cols}
print("Selected model score columns:", score_cols)

pred_df = add_score_normalizations(pred_df, score_cols)
pred_df["rrf_score"] = rrf_score(pred_df, score_cols, k=60)
rrf_ndcg = compute_ndcg_fast(pred_df, "rrf_score")
print(f"RRF NDCG@5 diagnostic: {rrf_ndcg:.6f}")

z_cols = [f"{col}_z" for col in score_cols]
weights, opt_ndcg = optimize_blend_weights(pred_df, z_cols)
pred_df["opt_z_blend"] = sum(w * pred_df[col] for w, col in zip(weights, z_cols))
opt_z_oof_ndcg, opt_z_oof_weights = evaluate_oof_blend(pred_df, z_cols, signed=False, n_splits=META_N_SPLITS)
print("Optimized nonnegative z-blend in-sample:", opt_ndcg, dict(zip(score_cols, weights.round(4))))
print(f"Optimized nonnegative z-blend OOF: {opt_z_oof_ndcg:.6f}")

signed_weights, signed_ndcg = optimize_signed_blend_weights(pred_df, z_cols)
pred_df["signed_z_blend"] = sum(w * pred_df[col] for w, col in zip(signed_weights, z_cols))
signed_oof_ndcg, signed_oof_weights = evaluate_oof_blend(pred_df, z_cols, signed=True, n_splits=META_N_SPLITS)
print("Optimized signed z-blend in-sample:", signed_ndcg, dict(zip(score_cols, signed_weights.round(4))))
print(f"Optimized signed z-blend OOF: {signed_oof_ndcg:.6f}")

segment_blend_ndcg = None
segment_blend_config = None
if RUN_SEGMENT_BLEND and SEGMENT_BLEND_COL in pred_df.columns:
    segment_blend_ndcg, segment_blend_config, segment_fold_configs = segment_blend_oof(
        pred_df,
        z_cols,
        SEGMENT_BLEND_COL,
        shrinkage=SEGMENT_SHRINKAGE,
        n_splits=META_N_SPLITS,
    )
    if segment_blend_config is not None:
        pred_df["segment_signed_z_blend"] = apply_segment_blend(pred_df, segment_blend_config)
        print(f"Segment signed-z OOF: {segment_blend_ndcg:.6f} using {SEGMENT_BLEND_COL}")
else:
    segment_fold_configs = []
    print("Skipping segment blend")

meta_model_full = None
meta_feature_cols = []
meta_oof_ndcg = None
meta_full_rounds = None
meta_fold_scores = []
residual_meta_ndcg = None
residual_meta_alpha = None
residual_meta_fold_alphas = []
residual_meta_base_col = None
if RUN_META_STACKER:
    meta_df, meta_feature_cols = prepare_meta_frame(pred_df, score_cols, routing_cols)
    meta_oof, meta_oof_ndcg, meta_model_full, meta_full_rounds, meta_fold_scores = train_meta_oof(
        meta_df,
        meta_feature_cols,
    )
    pred_df["meta_oof_score"] = meta_oof
    print(f"Meta OOF NDCG@5: {meta_oof_ndcg:.6f}  final_rounds={meta_full_rounds}")
    print("Meta features:", meta_feature_cols)

    residual_base_candidates = {
        "signed_z_blend": float(signed_oof_ndcg),
        "rrf_score": float(rrf_ndcg),
    }
    if segment_blend_ndcg is not None and np.isfinite(segment_blend_ndcg):
        residual_base_candidates["segment_signed_z_blend"] = float(segment_blend_ndcg)
    residual_meta_base_col = max(residual_base_candidates, key=residual_base_candidates.get)
    residual_meta_ndcg, residual_meta_alpha, residual_meta_fold_alphas = residual_meta_oof(
        pred_df,
        residual_meta_base_col,
        "meta_oof_score",
        RESIDUAL_META_ALPHAS,
        n_splits=META_N_SPLITS,
    )
    pred_df["residual_meta_score"] = residual_meta_score(
        pred_df,
        residual_meta_base_col,
        "meta_oof_score",
        residual_meta_alpha,
    )
    print(
        f"Residual meta OOF: {residual_meta_ndcg:.6f} base={residual_meta_base_col} "
        f"alpha={residual_meta_alpha} fold_alphas={residual_meta_fold_alphas}"
    )
else:
    print("Skipping meta stacker")


## Diagnostics And Selector Payload

This cell preserves the standard Velocity-style diagnostics. The next Apex gate cell then applies stricter paired-bootstrap, segment-diversity, and residual-selection rules before final retraining.


In [ ]:
ensemble_scores_in_sample = {
    "rrf_score": compute_ndcg_fast(pred_df, "rrf_score"),
    "opt_z_blend": compute_ndcg_fast(pred_df, "opt_z_blend"),
    "signed_z_blend": compute_ndcg_fast(pred_df, "signed_z_blend"),
}
blend_oof_scores = {
    "opt_z_blend": float(opt_z_oof_ndcg),
    "signed_z_blend": float(signed_oof_ndcg),
}
selector_scores = {
    "signed_z_blend": float(signed_oof_ndcg),
    "rrf_score": float(rrf_ndcg),
}
if segment_blend_ndcg is not None and np.isfinite(segment_blend_ndcg):
    base_before_segment = max(selector_scores.values())
    if segment_blend_ndcg >= base_before_segment + SEGMENT_BLEND_MIN_GAIN:
        selector_scores["segment_signed_z_blend"] = float(segment_blend_ndcg)
if residual_meta_ndcg is not None and np.isfinite(residual_meta_ndcg):
    meta_fold_std = float(np.std(meta_fold_scores)) if meta_fold_scores else np.inf
    meta_fold_min = float(np.min(meta_fold_scores)) if meta_fold_scores else -np.inf
    residual_gain = float(residual_meta_ndcg - max(selector_scores.values()))
    residual_gate = meta_fold_std < 0.0055 and meta_fold_min >= 0.405 and residual_gain >= RESIDUAL_META_MIN_GAIN
    if residual_gate:
        selector_scores["residual_meta_score"] = float(residual_meta_ndcg)
else:
    meta_fold_std = None
    meta_fold_min = None
    residual_gain = None
    residual_gate = False

best_ensemble_col = max(selector_scores, key=selector_scores.get)
best_ensemble_ndcg = selector_scores[best_ensemble_col]
passes_submission_gate = bool(best_ensemble_ndcg >= SUBMISSION_GATE)
submission_recommendation = "submit" if passes_submission_gate else "do_not_submit"

corr_cols = [c for c in score_cols + ["signed_z_blend", "rrf_score", "segment_signed_z_blend", "meta_oof_score", "residual_meta_score"] if c in pred_df.columns]
rank_correlation = pred_df[corr_cols].corr(method="spearman").round(6).to_dict() if len(corr_cols) > 1 else {}
top5_overlap_diagnostics = {}
for other in ["rrf_score", "segment_signed_z_blend", "meta_oof_score", "residual_meta_score"]:
    if other in pred_df.columns and other != "signed_z_blend":
        top5_overlap_diagnostics[f"{other}_vs_signed_z"] = top5_overlap(pred_df, other, "signed_z_blend")

segment_cols = [
    "is_domestic",
    "is_family",
    "is_last_minute",
    "is_advance_purchase",
    "is_leisure_search",
]
segment_diagnostics = {best_ensemble_col: segment_ndcg(pred_df, best_ensemble_col, segment_cols)}
if best_ensemble_col != "signed_z_blend":
    segment_diagnostics["signed_z_blend"] = segment_ndcg(pred_df, "signed_z_blend", segment_cols)

print("Selector scores:")
for col, score in sorted(selector_scores.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {col}: {score:.6f}")
print("OOF blend diagnostics:")
for col, score in sorted(blend_oof_scores.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {col}: {score:.6f}")
print("In-sample diagnostics only:")
for col, score in sorted(ensemble_scores_in_sample.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {col}: {score:.6f}")
print(
    f"Meta/residual gate: residual_gate={residual_gate}, fold_std={meta_fold_std}, "
    f"min_fold={meta_fold_min}, residual_gain={residual_gain}"
)
print(
    f"Selected ensemble: {best_ensemble_col}  selection_score={best_ensemble_ndcg:.6f}  "
    f"gate={SUBMISSION_GATE:.5f} recommendation={submission_recommendation}"
)

best_iterations = {
    "pointwise": lgb_iteration_count(pw_model, PW_NUM_BOOST_ROUND),
    "click": None if "click_score" not in score_cols else lgb_iteration_count(click_model, CLICK_NUM_BOOST_ROUND),
    "lambdarank": None if "lr_score" not in score_cols else lgb_iteration_count(lr_model, LR_NUM_BOOST_ROUND),
    "lambdarank_trunc5": None if "lr_trunc5_score" not in score_cols else lgb_iteration_count(lr_trunc5_model, LR_TRUNC5_NUM_BOOST_ROUND),
    "rank_xendcg_seeds": {
        str(seed): lgb_iteration_count(xendcg_models.get(seed), XENDCG_NUM_BOOST_ROUND)
        for seed in XENDCG_SEEDS
        if seed in xendcg_models
    },
    "catboost": None if cat_model is None or "cb_score" not in score_cols else catboost_iteration_count(cat_model, CATBOOST_ITERATIONS),
}

run_payload = {
    "validation_tag": VALIDATION_TAG,
    "use_sample": USE_SAMPLE,
    "sample_frac": SAMPLE_FRAC if USE_SAMPLE else None,
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "n_features": int(len(feature_cols)),
    "n_meta_routing_features": int(len(routing_cols)),
    "feature_cols": feature_cols,
    "routing_cols": routing_cols,
    "gain_policy": GAIN_POLICY,
    "xendcg_seeds": XENDCG_SEEDS,
    "candidate_score_cols": candidate_score_cols,
    "score_cols": score_cols,
    "candidate_base_scores": {k: float(v) for k, v in candidate_base_scores.items()},
    "base_scores": {k: float(v) for k, v in base_scores.items()},
    "xendcg_seed_scores": xendcg_seed_scores,
    "lomo_diagnostics": lomo_diagnostics,
    "rrf_ndcg5": float(rrf_ndcg),
    "opt_z_ndcg5_in_sample": float(opt_ndcg),
    "opt_z_ndcg5_oof": float(opt_z_oof_ndcg),
    "opt_z_weights": dict(zip(score_cols, [float(w) for w in weights])),
    "signed_z_ndcg5_in_sample": float(signed_ndcg),
    "signed_z_ndcg5_oof": float(signed_oof_ndcg),
    "signed_z_weights": dict(zip(score_cols, [float(w) for w in signed_weights])),
    "segment_blend_ndcg5_oof": None if segment_blend_ndcg is None else float(segment_blend_ndcg),
    "segment_blend_config": segment_blend_config,
    "meta_oof_ndcg5": None if meta_oof_ndcg is None else float(meta_oof_ndcg),
    "meta_full_rounds": None if meta_full_rounds is None else int(meta_full_rounds),
    "meta_fold_scores": [float(x) for x in meta_fold_scores],
    "meta_fold_std": None if meta_fold_std is None or not np.isfinite(meta_fold_std) else float(meta_fold_std),
    "meta_fold_min": None if meta_fold_min is None or not np.isfinite(meta_fold_min) else float(meta_fold_min),
    "meta_feature_cols": meta_feature_cols,
    "residual_meta_ndcg5_oof": None if residual_meta_ndcg is None else float(residual_meta_ndcg),
    "residual_meta_alpha": residual_meta_alpha,
    "residual_meta_base_col": residual_meta_base_col,
    "residual_meta_fold_alphas": residual_meta_fold_alphas,
    "residual_meta_gate": bool(residual_gate),
    "selector_scores": {k: float(v) for k, v in selector_scores.items()},
    "best_ensemble_col": best_ensemble_col,
    "best_ensemble_ndcg5": float(best_ensemble_ndcg),
    "submission_gate": float(SUBMISSION_GATE),
    "passes_submission_gate": bool(passes_submission_gate),
    "submission_recommendation": submission_recommendation,
    "rank_correlation": rank_correlation,
    "top5_overlap_diagnostics": top5_overlap_diagnostics,
    "segment_diagnostics": segment_diagnostics,
    "best_iterations": best_iterations,
}
save_run_json(OUTPUT_DIR / VALIDATION_RUN_NAME, run_payload)
pred_df.to_parquet(OUTPUT_DIR / VALIDATION_PREDICTIONS_NAME, index=False)
run_payload


## Apex Evidence Gate

This gate protects Velocity. It rejects simple segment blends that are nearly identical to signed-z, and it promotes residual meta only when paired bootstrap and fold stability agree.


In [ ]:
def per_search_ndcg(frame, score_col, k=5):
    tmp = frame[["srch_id", score_col, "booking_bool", "click_bool"]].copy()
    tmp["_rel"] = tmp["booking_bool"] * 5 + (tmp["click_bool"] - tmp["booking_bool"]).clip(lower=0)
    tmp["_neg_score"] = -tmp[score_col]
    tmp["_rank"] = tmp.groupby("srch_id")["_neg_score"].rank(method="first").astype(int)
    top_k = tmp[tmp["_rank"] <= k].copy()
    top_k["_dcg"] = top_k["_rel"] / np.log2(top_k["_rank"] + 1)
    dcg = top_k.groupby("srch_id")["_dcg"].sum()

    ideal = tmp.sort_values(["srch_id", "_rel"], ascending=[True, False]).copy()
    ideal["_ideal_rank"] = ideal.groupby("srch_id").cumcount() + 1
    ideal = ideal[ideal["_ideal_rank"] <= k]
    ideal["_idcg"] = ideal["_rel"] / np.log2(ideal["_ideal_rank"] + 1)
    idcg = ideal.groupby("srch_id")["_idcg"].sum()
    ndcg = (dcg / idcg).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return ndcg.reindex(tmp["srch_id"].drop_duplicates().to_numpy(), fill_value=0.0)


def paired_bootstrap_summary(frame, candidate_col, baseline_col, n_boot=2000, seed=RANDOM_STATE):
    cand = per_search_ndcg(frame, candidate_col)
    base = per_search_ndcg(frame, baseline_col)
    delta = (cand - base).dropna().to_numpy(dtype="float64")
    if len(delta) == 0:
        return {"mean_delta": np.nan, "p05": np.nan, "p50": np.nan, "p95": np.nan, "p_positive": np.nan, "p_gt_00025": np.nan}
    rng = np.random.default_rng(seed)
    draws = rng.choice(delta, size=(n_boot, len(delta)), replace=True).mean(axis=1)
    return {
        "mean_delta": float(delta.mean()),
        "p05": float(np.quantile(draws, 0.05)),
        "p50": float(np.quantile(draws, 0.50)),
        "p95": float(np.quantile(draws, 0.95)),
        "p_positive": float((draws > 0).mean()),
        "p_gt_00025": float((draws > 0.00025).mean()),
    }


def segment_value_score(frame, score_col, segment_col, value):
    if segment_col not in frame.columns:
        return np.nan
    search_values = frame.groupby("srch_id")[segment_col].first()
    ids = search_values.index[search_values == value]
    if len(ids) == 0:
        return np.nan
    return float(compute_ndcg_fast(frame[frame["srch_id"].isin(ids)], score_col))


def oof_blend_predictions(frame, normalized_score_cols, signed=True, n_splits=META_N_SPLITS):
    splits = min(n_splits, frame["srch_id"].nunique())
    if splits < 2:
        return np.full(len(frame), np.nan, dtype="float32"), np.nan, []
    oof = np.zeros(len(frame), dtype="float32")
    fold_weights = []
    splitter = GroupKFold(n_splits=splits)
    for fit_idx, hold_idx in splitter.split(frame, groups=frame["srch_id"]):
        fit_frame = frame.iloc[fit_idx]
        hold_frame = frame.iloc[hold_idx]
        if signed:
            weights, _ = optimize_signed_blend_weights(fit_frame, normalized_score_cols, maxiter=250)
        else:
            weights, _ = optimize_blend_weights(fit_frame, normalized_score_cols, maxiter=250)
        fold_weights.append(weights)
        blend = np.zeros(len(hold_frame), dtype="float64")
        for weight, col in zip(weights, normalized_score_cols):
            blend += weight * hold_frame[col].to_numpy(dtype="float64")
        oof[hold_idx] = blend.astype("float32")
    tmp = frame[["srch_id", "booking_bool", "click_bool"]].copy()
    tmp["_oof"] = oof
    return oof, float(compute_ndcg_fast(tmp, "_oof")), fold_weights


def segment_blend_oof_predictions(frame, z_cols, segment_col, shrinkage=SEGMENT_SHRINKAGE, n_splits=META_N_SPLITS):
    if segment_col not in frame.columns:
        return np.full(len(frame), np.nan, dtype="float32"), np.nan
    splits = min(n_splits, frame["srch_id"].nunique())
    if splits < 2:
        return np.full(len(frame), np.nan, dtype="float32"), np.nan
    oof = np.zeros(len(frame), dtype="float32")
    splitter = GroupKFold(n_splits=splits)
    for fit_idx, hold_idx in splitter.split(frame, groups=frame["srch_id"]):
        fit_frame = frame.iloc[fit_idx]
        hold_frame = frame.iloc[hold_idx]
        global_w, _ = optimize_signed_blend_weights(fit_frame, z_cols, maxiter=250)
        segment_weights = {}
        for value in sorted(fit_frame[segment_col].dropna().unique()):
            fit_seg = fit_frame[fit_frame[segment_col] == value]
            if fit_seg["srch_id"].nunique() < 10:
                seg_w = global_w
            else:
                raw_w, _ = optimize_signed_blend_weights(fit_seg, z_cols, maxiter=250)
                seg_w = (1.0 - shrinkage) * global_w + shrinkage * raw_w
                denom = np.abs(seg_w).sum()
                seg_w = global_w if denom < 1e-12 else seg_w / denom
            segment_weights[str(value)] = seg_w
        for value, sub_idx in hold_frame.groupby(segment_col).groups.items():
            idx = np.asarray(list(sub_idx))
            weights = segment_weights.get(str(value), global_w)
            sub = frame.loc[idx]
            blend = np.zeros(len(sub), dtype="float64")
            for weight, col in zip(weights, z_cols):
                blend += weight * sub[col].to_numpy(dtype="float64")
            oof[idx] = blend.astype("float32")
    tmp = frame[["srch_id", "booking_bool", "click_bool"]].copy()
    tmp["_segment_oof"] = oof
    return oof, float(compute_ndcg_fast(tmp, "_segment_oof"))


def residual_meta_oof_predictions(frame, base_col, meta_col, alphas, n_splits=META_N_SPLITS):
    splits = min(n_splits, frame["srch_id"].nunique())
    if splits < 2:
        return np.full(len(frame), np.nan, dtype="float32"), np.nan, None, []
    oof = np.zeros(len(frame), dtype="float32")
    fold_alphas = []
    splitter = GroupKFold(n_splits=splits)
    for fit_idx, hold_idx in splitter.split(frame, groups=frame["srch_id"]):
        fit = frame.iloc[fit_idx].copy()
        hold = frame.iloc[hold_idx].copy()
        best_alpha = None
        best_score = -np.inf
        for alpha in alphas:
            fit["_residual"] = residual_meta_score(fit, base_col, meta_col, alpha)
            score = compute_ndcg_fast(fit, "_residual")
            if score > best_score:
                best_score = score
                best_alpha = alpha
        hold["_residual"] = residual_meta_score(hold, base_col, meta_col, best_alpha)
        oof[hold_idx] = hold["_residual"].to_numpy(dtype="float32")
        fold_alphas.append(float(best_alpha))
    tmp = frame[["srch_id", "booking_bool", "click_bool"]].copy()
    tmp["_residual_oof"] = oof
    score = float(compute_ndcg_fast(tmp, "_residual_oof"))
    full_scores = []
    for alpha in alphas:
        tmp_full = frame.copy()
        tmp_full["_residual"] = residual_meta_score(tmp_full, base_col, meta_col, alpha)
        full_scores.append((compute_ndcg_fast(tmp_full, "_residual"), alpha))
    _, full_alpha = max(full_scores, key=lambda x: x[0])
    return oof, score, float(full_alpha), fold_alphas


# Build true OOF columns for selection. Full-score columns remain for final configuration only.
apex_oof_score_cols = {}
apex_oof_scores = {}
base_oof_col = "apex_oof_signed_z_blend"
pred_df[base_oof_col], apex_oof_scores["signed_z_blend"], apex_signed_fold_weights = oof_blend_predictions(
    pred_df, z_cols, signed=True, n_splits=META_N_SPLITS
)
apex_oof_score_cols["signed_z_blend"] = base_oof_col

if "rrf_score" in pred_df.columns:
    apex_oof_score_cols["rrf_score"] = "rrf_score"
    apex_oof_scores["rrf_score"] = float(rrf_ndcg)

if RUN_SEGMENT_BLEND and SEGMENT_BLEND_COL in pred_df.columns:
    seg_oof_col = "apex_oof_segment_signed_z_blend"
    pred_df[seg_oof_col], apex_oof_scores["segment_signed_z_blend"] = segment_blend_oof_predictions(
        pred_df, z_cols, SEGMENT_BLEND_COL, shrinkage=SEGMENT_SHRINKAGE, n_splits=META_N_SPLITS
    )
    apex_oof_score_cols["segment_signed_z_blend"] = seg_oof_col

if RUN_META_STACKER and "meta_oof_score" in pred_df.columns:
    residual_base_for_gate = residual_meta_base_col or "signed_z_blend"
    res_oof_col = "apex_oof_residual_meta_score"
    pred_df[res_oof_col], apex_oof_scores["residual_meta_score"], apex_residual_full_alpha, apex_residual_fold_alphas = residual_meta_oof_predictions(
        pred_df, residual_base_for_gate, "meta_oof_score", RESIDUAL_META_ALPHAS, n_splits=META_N_SPLITS
    )
    apex_oof_score_cols["residual_meta_score"] = res_oof_col

apex_baseline_col = "signed_z_blend"
apex_baseline_oof_col = apex_oof_score_cols[apex_baseline_col]
apex_baseline_score = float(apex_oof_scores[apex_baseline_col])

apex_gate_diagnostics = {}
apex_passed = {}
for logical_col, oof_col in apex_oof_score_cols.items():
    score = float(apex_oof_scores[logical_col])
    boot = paired_bootstrap_summary(pred_df, oof_col, apex_baseline_oof_col)
    corr = None
    overlap = None
    if logical_col != apex_baseline_col:
        corr = float(pred_df[[oof_col, apex_baseline_oof_col]].corr(method="spearman").iloc[0, 1])
        overlap = float(top5_overlap(pred_df, oof_col, apex_baseline_oof_col))
    intl = segment_value_score(pred_df, oof_col, "is_domestic", 0)
    dom = segment_value_score(pred_df, oof_col, "is_domestic", 1)
    apex_gate_diagnostics[logical_col] = {
        "oof_score_col": oof_col,
        "score": score,
        "delta_vs_apex_baseline_col": float(score - apex_baseline_score),
        "bootstrap_vs_apex_baseline_col": boot,
        "spearman_vs_apex_baseline_col": corr,
        "top5_overlap_vs_apex_baseline_col": overlap,
        "international_ndcg": intl,
        "domestic_ndcg": dom,
    }

    if logical_col == apex_baseline_col:
        passed = True
        reason = "protected local baseline inside this run"
    elif logical_col == "rrf_score":
        passed = (score >= apex_baseline_score + 0.00025 and boot["p_positive"] >= 0.85)
        reason = "RRF must clear OOF and bootstrap"
    elif logical_col == "segment_signed_z_blend":
        baseline_intl = segment_value_score(pred_df, apex_baseline_oof_col, "is_domestic", 0)
        passed = (
            score >= apex_baseline_score + SEGMENT_BLEND_MIN_GAIN
            and corr is not None and corr < SEGMENT_SPEARMAN_MAX
            and overlap is not None and overlap < SEGMENT_TOP5_OVERLAP_MAX
            and (np.isnan(intl) or np.isnan(baseline_intl) or intl >= baseline_intl)
        )
        reason = "simple segment blend must create real OOF rank movement"
    elif logical_col == "residual_meta_score":
        meta_fold_std_local = float(np.std(meta_fold_scores)) if meta_fold_scores else np.inf
        meta_fold_min_local = float(np.min(meta_fold_scores)) if meta_fold_scores else -np.inf
        passed = (
            score >= apex_baseline_score + RESIDUAL_META_MIN_GAIN
            and boot["p_positive"] >= BOOTSTRAP_MIN_P_POSITIVE
            and meta_fold_std_local < 0.0055
            and meta_fold_min_local >= 0.405
        )
        reason = "residual meta must clear lift, bootstrap, and meta fold stability"
    else:
        passed = False
        reason = "unknown candidate class"
    apex_passed[logical_col] = {"passed": bool(passed), "reason": reason}

passed_cols = [col for col, info in apex_passed.items() if info["passed"]]
if not passed_cols:
    passed_cols = [apex_baseline_col]
apex_selected_col = max(passed_cols, key=lambda c: apex_gate_diagnostics[c]["score"])
apex_selected_score = apex_gate_diagnostics[apex_selected_col]["score"]
passes_submission_gate = bool(apex_selected_score >= SUBMISSION_GATE)
strong_submission_gate = bool(apex_selected_score >= STRONG_SUBMISSION_GATE)
submission_recommendation = "submit" if passes_submission_gate else "do_not_submit"

best_ensemble_col = apex_selected_col
best_ensemble_ndcg = apex_selected_score
selector_scores = {col: apex_gate_diagnostics[col]["score"] for col in apex_oof_score_cols if apex_passed.get(col, {}).get("passed")}
if best_ensemble_col not in selector_scores:
    selector_scores[best_ensemble_col] = best_ensemble_ndcg

run_payload.update({
    "validation_tag": VALIDATION_TAG,
    "velocity_protected_public": VELOCITY_PROTECTED_PUBLIC,
    "velocity_protected_oof": VELOCITY_PROTECTED_OOF,
    "apex_baseline_col": apex_baseline_col,
    "apex_baseline_oof_col": apex_baseline_oof_col,
    "apex_baseline_score": apex_baseline_score,
    "apex_oof_scores": {k: float(v) for k, v in apex_oof_scores.items()},
    "apex_gate_diagnostics": apex_gate_diagnostics,
    "apex_gate_passed": apex_passed,
    "best_ensemble_col": best_ensemble_col,
    "best_ensemble_ndcg5": float(best_ensemble_ndcg),
    "selector_scores": {k: float(v) for k, v in selector_scores.items()},
    "submission_gate": float(SUBMISSION_GATE),
    "strong_submission_gate": float(STRONG_SUBMISSION_GATE),
    "passes_submission_gate": bool(passes_submission_gate),
    "passes_strong_submission_gate": bool(strong_submission_gate),
    "submission_recommendation": submission_recommendation,
})
save_run_json(OUTPUT_DIR / VALIDATION_RUN_NAME, run_payload)
pred_df.to_parquet(OUTPUT_DIR / VALIDATION_PREDICTIONS_NAME, index=False)

print("Apex gate candidates using OOF score columns:")
for col, diag in sorted(apex_gate_diagnostics.items(), key=lambda kv: kv[1]["score"], reverse=True):
    passed = apex_passed[col]["passed"]
    print(
        f"  {col}: score={diag['score']:.6f} delta={diag['delta_vs_apex_baseline_col']:+.6f} "
        f"p_pos={diag['bootstrap_vs_apex_baseline_col']['p_positive']:.3f} passed={passed}"
    )
print(
    f"Selected Apex candidate: {best_ensemble_col} score={best_ensemble_ndcg:.6f} "
    f"gate={SUBMISSION_GATE:.6f} recommendation={submission_recommendation}"
)
run_payload


## Optional Final Submission Retrain


In [ ]:
def fit_final_rank_model(score_col, params, num_boost_round):
    full_sorted, X_full, y_full, full_group = sort_for_group_model(
        final_train_df, final_feature_cols, make_relevance(final_train_df)
    )
    test_sorted, X_test, _, _ = sort_for_group_model(test_df, final_feature_cols)
    model = train_lgb(
        params,
        lgb.Dataset(X_full, label=y_full, group=full_group),
        num_boost_round=num_boost_round,
    )
    pred = test_sorted[["srch_id", "prop_id"]].copy()
    pred[score_col] = model.predict(X_test, num_iteration=lgb_iteration_count(model, num_boost_round))
    return model, pred


def train_final_catboost(score_col, catboost_iters):
    cb_feature_cols = get_feature_cols(final_train_df, keep_svd_factors=CATBOOST_KEEP_SVD_FACTORS)
    cb_full, X_cb_full, y_cb_full, _ = sort_for_group_model(final_train_df, cb_feature_cols, make_relevance(final_train_df))
    cb_test, X_cb_test, _, _ = sort_for_group_model(test_df, cb_feature_cols)
    cat_features_names = []
    if CATBOOST_USE_ID_CATEGORICALS:
        cat_features_names.extend(["prop_id", "srch_destination_id"])
    cat_features_names.extend(["site_id", "visitor_location_country_id", "prop_country_id"])
    cat_features_names = list(dict.fromkeys(cat_features_names))
    missing_cat_features = set(cat_features_names) - set(cb_feature_cols)
    assert not missing_cat_features, f"Missing CatBoost categoricals: {missing_cat_features}"
    cat_features_idx = [cb_feature_cols.index(c) for c in cat_features_names]
    cb_full_pool = Pool(
        X_cb_full,
        label=y_cb_full,
        group_id=cb_full["srch_id"].to_numpy(),
        cat_features=cat_features_idx,
        feature_names=cb_feature_cols,
    )
    cb_test_pool = Pool(
        X_cb_test,
        group_id=cb_test["srch_id"].to_numpy(),
        cat_features=cat_features_idx,
        feature_names=cb_feature_cols,
    )
    final_cb = CatBoostRanker(
        loss_function="YetiRank",
        eval_metric="NDCG:top=5;type=Base",
        iterations=catboost_iters,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=RANDOM_STATE,
        task_type=CATBOOST_TASK_TYPE,
        verbose=200,
        allow_writing_files=False,
    )
    final_cb.fit(cb_full_pool)
    pred = cb_test[["srch_id", "prop_id"]].copy()
    pred[score_col] = final_cb.predict(cb_test_pool)
    return final_cb, pred


def load_or_train_meta_model(validation_payload):
    in_memory = globals().get("meta_model_full")
    if in_memory is not None:
        return in_memory
    meta_feature_cols_local = validation_payload.get("meta_feature_cols", [])
    saved_pred_df = pd.read_parquet(OUTPUT_DIR / VALIDATION_PREDICTIONS_NAME)
    missing_meta = set(meta_feature_cols_local) - set(saved_pred_df.columns)
    assert not missing_meta, f"Saved validation predictions missing meta features: {missing_meta}"
    saved_pred_df[meta_feature_cols_local] = saved_pred_df[meta_feature_cols_local].replace([np.inf, -np.inf], np.nan)
    _, X_meta_full, y_meta_full, meta_group_full = sort_for_group_model(
        saved_pred_df,
        meta_feature_cols_local,
        make_relevance(saved_pred_df),
    )
    return train_lgb(
        with_lgb_seed(META_PARAMS, RANDOM_STATE + 999),
        lgb.Dataset(X_meta_full, label=y_meta_full, group=meta_group_full),
        num_boost_round=int(validation_payload.get("meta_full_rounds") or 300),
    )


if not RUN_FINAL_SUBMISSION:
    print("RUN_FINAL_SUBMISSION=False; validation-only run complete.")
else:
    validation_payload = globals().get("run_payload")
    if validation_payload is None:
        with (OUTPUT_DIR / VALIDATION_RUN_NAME).open() as f:
            validation_payload = json.load(f)
    assert validation_payload.get("passes_submission_gate") or ALLOW_DIAGNOSTIC_SUBMISSION, (
        f"Selected validation score did not pass the {SUBMISSION_GATE:.5f} gate. "
        "Set ALLOW_DIAGNOSTIC_SUBMISSION=True only for a deliberate leaderboard diagnostic."
    )
    assert FINAL_SUBMISSION_NAME not in {"submission_afterburner.csv", "submission_overdrive.csv"}

    score_cols = list(validation_payload["score_cols"])
    best_iterations = dict(validation_payload.get("best_iterations", {}))
    best_ensemble_col = validation_payload["best_ensemble_col"]
    signed_weights_final = np.array([validation_payload["signed_z_weights"][c] for c in score_cols], dtype="float64")
    segment_config_final = validation_payload.get("segment_blend_config")
    meta_feature_cols_final = validation_payload.get("meta_feature_cols", [])

    for _name in [
        "train_df", "val_df", "train_sorted", "val_sorted", "pred_df", "meta_df",
    ]:
        if _name in globals():
            del globals()[_name]
    gc.collect()

    full_train_raw = load_train(sample_frac=None, random_state=RANDOM_STATE)
    test_raw = load_test()
    if USE_TRAIN_TEST_COUNTS:
        count_reference_frame = pd.concat(
            [
                full_train_raw[["prop_id", "srch_destination_id"]],
                test_raw[["prop_id", "srch_destination_id"]],
            ],
            ignore_index=True,
        )
    else:
        count_reference_frame = full_train_raw

    final_train_df, test_df = build_test_feature_frame(
        full_train_raw,
        test_raw,
        count_reference_frame=count_reference_frame,
        svd_components=SVD_COMPONENTS,
    )
    final_train_df, test_df = add_apex_features(final_train_df, test_df, n_splits=TARGET_PRIOR_SPLITS)
    del full_train_raw, test_raw, count_reference_frame
    gc.collect()

    final_feature_cols = get_feature_cols(final_train_df)
    final_routing_cols = available(META_ROUTING_CANDIDATES, test_df)
    final_scores = test_df[["srch_id", "prop_id"] + final_routing_cols].copy()

    if "pw_score" in score_cols:
        final_pw = train_lgb(
            PW_PARAMS,
            lgb.Dataset(final_train_df[final_feature_cols], label=final_train_df["booking_bool"].to_numpy()),
            num_boost_round=iteration_or_fallback(best_iterations.get("pointwise"), PW_NUM_BOOST_ROUND),
        )
        final_scores["pw_score"] = final_pw.predict(test_df[final_feature_cols])

    if "click_score" in score_cols:
        final_click = train_lgb(
            CLICK_PARAMS,
            lgb.Dataset(final_train_df[final_feature_cols], label=final_train_df["click_bool"].to_numpy()),
            num_boost_round=iteration_or_fallback(best_iterations.get("click"), CLICK_NUM_BOOST_ROUND),
        )
        final_scores["click_score"] = final_click.predict(test_df[final_feature_cols])

    if "lr_score" in score_cols:
        _, pred = fit_final_rank_model(
            "lr_score",
            LR_PARAMS,
            iteration_or_fallback(best_iterations.get("lambdarank"), LR_NUM_BOOST_ROUND),
        )
        final_scores = final_scores.merge(pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")

    if "lr_trunc5_score" in score_cols:
        _, pred = fit_final_rank_model(
            "lr_trunc5_score",
            LR_TRUNC5_PARAMS,
            iteration_or_fallback(best_iterations.get("lambdarank_trunc5"), LR_TRUNC5_NUM_BOOST_ROUND),
        )
        final_scores = final_scores.merge(pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")

    if "xendcg_avg_score" in score_cols:
        seed_score_cols = []
        seed_iters = best_iterations.get("rank_xendcg_seeds", {})
        for seed in XENDCG_SEEDS:
            col = f"xendcg_s{seed}_score"
            params = with_lgb_seed(XENDCG_PARAMS, seed)
            _, pred = fit_final_rank_model(
                col,
                params,
                iteration_or_fallback(seed_iters.get(str(seed)), XENDCG_NUM_BOOST_ROUND),
            )
            final_scores = final_scores.merge(pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")
            z_col = f"{col}_seed_z"
            final_scores[z_col] = z_normalize_per_search(final_scores, col)
            seed_score_cols.append(z_col)
        final_scores["xendcg_avg_score"] = final_scores[seed_score_cols].mean(axis=1).astype("float32")

    if "cb_score" in score_cols:
        assert CatBoostRanker is not None and Pool is not None, "CatBoost required because validation selected cb_score"
        _, pred = train_final_catboost(
            "cb_score",
            iteration_or_fallback(best_iterations.get("catboost"), CATBOOST_ITERATIONS),
        )
        final_scores = final_scores.merge(pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")

    missing_scores = [c for c in score_cols if c not in final_scores.columns]
    assert not missing_scores, f"Final scores missing validation-selected models: {missing_scores}"
    final_scores = add_score_normalizations(final_scores, score_cols)
    final_z_cols = [f"{col}_z" for col in score_cols]

    if best_ensemble_col == "rrf_score":
        final_scores["final_score"] = rrf_score(final_scores, score_cols, k=60)
    elif best_ensemble_col == "signed_z_blend":
        final_scores["final_score"] = sum(w * final_scores[col] for w, col in zip(signed_weights_final, final_z_cols))
    elif best_ensemble_col == "segment_signed_z_blend":
        assert segment_config_final is not None
        final_scores["final_score"] = apply_segment_blend(final_scores, segment_config_final)
    elif best_ensemble_col == "residual_meta_score":
        meta_model = load_or_train_meta_model(validation_payload)
        missing = set(meta_feature_cols_final) - set(final_scores.columns)
        assert not missing, f"Missing final meta features: {missing}"
        final_scores[meta_feature_cols_final] = final_scores[meta_feature_cols_final].replace([np.inf, -np.inf], np.nan)
        final_scores["meta_oof_score"] = meta_model.predict(final_scores[meta_feature_cols_final])
        base_col = validation_payload["residual_meta_base_col"]
        if base_col == "segment_signed_z_blend" and base_col not in final_scores.columns:
            assert segment_config_final is not None
            final_scores[base_col] = apply_segment_blend(final_scores, segment_config_final)
        elif base_col == "rrf_score" and base_col not in final_scores.columns:
            final_scores[base_col] = rrf_score(final_scores, score_cols, k=60)
        elif base_col == "signed_z_blend" and base_col not in final_scores.columns:
            final_scores[base_col] = sum(w * final_scores[col] for w, col in zip(signed_weights_final, final_z_cols))
        final_scores["final_score"] = residual_meta_score(
            final_scores,
            base_col,
            "meta_oof_score",
            validation_payload["residual_meta_alpha"],
        )
    else:
        raise ValueError(f"Unknown selected ensemble: {best_ensemble_col}")

    sub_path = (ROOT / FINAL_SUBMISSION_NAME).resolve()
    if sub_path.exists() and not OVERWRITE_EXISTING_SUBMISSION:
        raise FileExistsError(
            f"{sub_path} already exists. Set OVERWRITE_EXISTING_SUBMISSION=True only after preserving the prior file."
        )
    sub = make_submission(final_scores, "final_score", sub_path)
    assert len(sub) == len(final_scores)
    assert sub["srch_id"].is_monotonic_increasing
    print(f"Wrote {sub_path} rows={len(sub):,}")
